In [5]:
import numpy as np
import pandas as pd
import scanpy as sc
import NaiveDE
import SpatialDE

# needs scipy==1.9.2

In [6]:

data_dir = '../data/Mouse_brain_MERFISH/'

age = '24wk'
data1="adata24wk_donor_id_10_slice_1"
data2="adata90wk_donor_id_5_slice_1"

sliceA = sc.read_h5ad(data_dir + data1 + ".h5ad")
sliceB = sc.read_h5ad(data_dir + data2 + ".h5ad")


In [7]:
filePath = '../local_data/PROMT'

# cosine_dist_gene_expr = np.load((f"{filePath}/cosine_dist_gene_expr_{data1}_{data2}.npy"))
cosine_dist_gene_expr = np.load(f"{filePath}/cosine_dist_gene_expr_{data1}_{data2}.npy")

# pi_mat = np.load(f"{filePath}/pi_matrix_{data1}_{data2}.npy")
pi_mat = np.load(f"{filePath}/pi_matrix_{data1}_{data2}.npy") 


# taking cost matrix as the all pair cosine_dist_gene_expr between slices
# mismatch_score_mat = pi_mat * cosine_dist_gene_expr
age_progression_score_mat = pi_mat * (cosine_dist_gene_expr)


sliceA.obs['age_progression_score'] = np.sum(age_progression_score_mat, axis=1, dtype=np.float64) / (1 / sliceA.n_obs) * 100
sliceB.obs['age_progression_score'] = np.sum(age_progression_score_mat, axis=0, dtype=np.float64) / (1 / sliceA.n_obs) * 100

In [8]:
if any(sliceA.obs['age_progression_score']) <= 0:
    # print the negative values
    print("There are negative values in the age progression")
    print(sliceA.obs['age_progression_score'][sliceA.obs['age_progression_score'] <= 0])

In [9]:
def filter_cells_by_count(adata, threshold):

    cell_type_counts = adata.obs['cell_type_annot'].value_counts()
    valid_cell_types = cell_type_counts[cell_type_counts >= threshold].index.tolist()

    return adata[adata.obs['cell_type_annot'].isin(valid_cell_types)]



In [15]:
def run_spatialde(sliceA, cell_type):
    counts = pd.DataFrame(sliceA.obs['age_progression_score'], columns=['age_progression_score'])
    # counts = counts.T[counts.sum(0) >= 3].T  # Filter practically unobserved genes

    print(counts.shape)
    sample_info = pd.DataFrame(sliceA.obsm['spatial'], columns=['x', 'y'])
    sample_info['total_counts'] = counts.to_numpy()

    norm_expr = NaiveDE.stabilize(counts.T).T
    resid_expr = NaiveDE.regress_out(sample_info, norm_expr.T, 'np.log(total_counts)').T

    resid_expr['log_total_count'] = np.log(sample_info['total_counts'])

    # X = sample_info[['x', 'y']]
    X = sample_info[['x', 'y']].to_numpy()
    results = SpatialDE.run(X, resid_expr)

    results.to_csv(f'./{age}/spatial_de_age_results_{cell_type}.csv', index=False)

In [11]:
def filter_cells_by_cell_type(adata, cell_type):
    filtered_adata = adata[adata.obs['cell_type_annot'] == cell_type].copy()
    return filtered_adata


In [16]:
sliceA = filter_cells_by_count(sliceA, 10)

import os
if not os.path.exists(f'./{age}'):
    os.makedirs(f'./{age}')

unique_cell_types = sliceA.obs['cell_type_annot'].unique()
for cell_type in unique_cell_types:
    print(f'\n\nRunning spatialDE for {cell_type}')
    sliceA_cell_type = filter_cells_by_cell_type(sliceA, cell_type)
    try:
        run_spatialde(sliceA_cell_type, cell_type)
    except:
        print(f'Error in running spatialDE for {cell_type}')
        continue




Running spatialDE for Olig
(1773, 1)


c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\scipy\optimize\_minpack_py.py:881: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',
Models: 100%|██████████| 10/10 [00:00<00:00, 48.03it/s]
c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\SpatialDE\base.py:310: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  model_results = model_results[model_results.groupby(['g'])['max_ll'].transform(max) == model_results['max_ll']]
c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\SpatialDE\util.py:19: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  pv = pv.ravel()  # flattens the array in place,



Running spatialDE for InN
(973, 1)


Models: 100%|██████████| 10/10 [00:00<00:00, 44.16it/s]
c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\SpatialDE\base.py:310: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  model_results = model_results[model_results.groupby(['g'])['max_ll'].transform(max) == model_results['max_ll']]
c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\scipy\optimize\_minpack_py.py:881: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',


Error in running spatialDE for InN


Running spatialDE for OPC
(215, 1)


...

Models: 100%|██████████| 10/10 [00:00<00:00, 46.23it/s]
c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\SpatialDE\base.py:310: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  model_results = model_results[model_results.groupby(['g'])['max_ll'].transform(max) == model_results['max_ll']]
c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\scipy\optimize\_minpack_py.py:881: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',


Error in running spatialDE for Macro


Running spatialDE for Peri
(295, 1)


Models: 100%|██████████| 10/10 [00:00<00:00, 59.18it/s]
c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\SpatialDE\base.py:310: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  model_results = model_results[model_results.groupby(['g'])['max_ll'].transform(max) == model_results['max_ll']]
c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\SpatialDE\util.py:19: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  pv = pv.ravel()  # flattens the array in place, more efficient than flatten()
c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\scipy\optimize\_minpack_py.py:881: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the param



Running spatialDE for Vlmc
(93, 1)


Models: 100%|██████████| 10/10 [00:00<00:00, 61.89it/s]
c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\SpatialDE\base.py:310: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  model_results = model_results[model_results.groupby(['g'])['max_ll'].transform(max) == model_results['max_ll']]
c:\Users\Anup\miniconda3\envs\thesis\lib\site-packages\SpatialDE\util.py:19: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  pv = pv.ravel()  # flattens the array in place, more efficient than flatten()
